# 재학습 모델 업데이트 & ECS 자동 재배포

## 목표

드리프트 감지 후 **이 노트북에서 직접** 재학습하고, 성능을 비교한 뒤,
새 모델을 **S3에 업로드**하여 ECS 서비스에 자동 반영하는 운영 흐름을 실습합니다.

### 전체 흐름

```
1. 드리프트 감지 (차시 27_2에서 완료)
2. 사람이 판단: "재학습이 필요한가?"
3. 이 노트북에서 재학습 → 성능 비교 → S3 업로드
4. S3 업로드 이벤트 → Lambda → ECS 자동 재시작
5. 새 컨테이너가 S3에서 최신 모델을 다운로드 → 서빙
```

### 핵심 포인트

- 드리프트 감지는 자동, **재학습 결정은 사람**이 한다
- 노트북에서 결과를 눈으로 확인하고 판단한 후 배포한다
- 모델 파일을 Docker 이미지에 넣지 않고 **S3에서 동적으로 로드**하면,
  이미지 재빌드 없이 모델만 교체할 수 있다

## 재학습 & 배포 아키텍처

```
┌──────────────────────────────────────────────────────────────────────────┐
│                    MLOps 모델 재학습 & 자동 배포                           │
├──────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  [1] PSI 드리프트 감지 (자동 — cron/Airflow)                              │
│       │                                                                  │
│       ▼                                                                  │
│  [2] Slack 알림 전송 (자동)                                               │
│       │                                                                  │
│       ▼                                                                  │
│  [3] 사람이 판단: "재학습이 필요한가?" ◀── Human-in-the-Loop               │
│       │                                                                  │
│       ▼                                                                  │
│  [4] 이 노트북에서 재학습 실행 (사람이 셀 실행)                             │
│       │                                                                  │
│       ├── [4a] 학습 데이터 + 예측 로그 로드                               │
│       ├── [4b] 새 모델 학습 (XGBoost)                                    │
│       ├── [4c] 기존 모델 vs 새 모델 AUC 비교                             │
│       └── [4d] 확인 후 S3에 모델 업로드                                   │
│                │                                                         │
│                ▼                                                         │
│  [5] S3 PUT 이벤트 → Lambda 트리거 (자동)                                │
│       │                                                                  │
│       ▼                                                                  │
│  [6] Lambda → ECS force-new-deployment (자동)                            │
│       │                                                                  │
│       ▼                                                                  │
│  [7] 새 컨테이너 시작 → S3에서 최신 모델 다운로드 → 서빙 시작 (자동)        │
│                                                                          │
└──────────────────────────────────────────────────────────────────────────┘

핵심: 사람은 [3]에서 "결정"하고 [4]에서 "실행"한다. 나머지는 전부 자동.
```

### 구성 요소

| 구성 요소 | 역할 | AWS 서비스 |
|-----------|------|-----------|
| 드리프트 감지 | PSI 계산 → 알림 | Python + CloudWatch |
| 이 노트북 | 재학습 + 비교 + S3 업로드 | Python + boto3 |
| 모델 저장소 | 최신 모델 파일 관리 | S3 |
| 자동 트리거 | S3 이벤트 → ECS 재시작 | Lambda |
| 서빙 인프라 | 새 모델로 예측 서비스 | ECS + FastAPI |

---

## 참고: Docker 이미지에 모델을 포함하는 방식 (과거 방식)

초기에는 모델 파일(pkl)을 **Docker 이미지 안에** 포함시키는 방법도 있습니다.

```
Dockerfile:
  COPY models/ ./models/    ← 이미지 빌드 시 모델이 포함됨
```

### 이 방식의 한계

| 항목 | 문제 |
|------|------|
| 모델 교체 | 모델만 바뀌어도 이미지 전체를 다시 빌드해야 함 |
| 빌드 시간 | docker build → push → ECS 배포까지 수 분 소요 |
| 이미지 크기 | 모델 파일 크기만큼 이미지가 커짐 |

### 우리의 방식: S3에서 모델을 동적으로 로드

```
과거 방식:  모델 → Docker 이미지 → ECR → ECS
            (모델 바뀔 때마다 이미지 재빌드)

현재 방식:  모델 → S3 버킷
            코드 → Docker 이미지 → ECR → ECS
            (서버 시작 시 S3에서 모델 다운로드)
```

---

## S3에서 모델 동적 로드 (현재 방식)

### S3 버킷 구조

```
s3://mlops-lab-models-{ACCOUNT_ID}/
  └── loan-api/
      ├── loan_pipeline.pkl        ← 최신 모델
      ├── label_encoders.pkl       ← 인코더
      └── feature_names.pkl        ← 피처 목록
```

### 장점

- 모델 교체 시 **이미지 재빌드가 필요 없다**
- S3에 새 모델 업로드 → ECS 재시작만 하면 끝
- 모델 버전 관리가 S3 버전 관리로 자동 처리됨
- 같은 이미지로 dev/staging/prod 환경별 다른 모델 사용 가능

---

## Step 1: 환경 설정

In [ ]:
!pip install boto3 xgboost -q

import os
import joblib
import boto3
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, classification_report
from xgboost import XGBClassifier

REGION = 'ap-northeast-2'
ACCOUNT_ID = boto3.client('sts').get_caller_identity()['Account']
BUCKET_NAME = f'mlops-lab-models-{ACCOUNT_ID}'
S3_PREFIX = 'loan-api'

print(f'AWS Account: {ACCOUNT_ID}')
print(f'S3 Bucket: {BUCKET_NAME}')

---

## 잠깐! 예측 로그를 학습에 써도 되는 건가?

재학습에서 가장 많이 혼동하는 부분입니다.
예측 로그의 `approved` 컬럼은 **기존 모델이 예측한 값**이지,
실제로 대출을 갚았는지/못 갚았는지의 **정답(Ground Truth)**이 아닙니다.

```
학습 데이터의 '승인여부'  = 실제 심사 결과 (Ground Truth)
예측 로그의 'approved'    = 기존 모델의 예측값 (Ground Truth 아님!)
```

### 그러면 왜 이 실습에서는 예측 로그를 학습에 사용하는가?

**이 실습은 MLOps 배포 파이프라인을 익히는 것이 목표**이기 때문입니다.
재학습 -> S3 업로드 -> ECS 재시작의 **운영 흐름**을 실습하는 것이 핵심이고,
모델 품질 자체는 이 차시의 주제가 아닙니다.

### 실무에서는 어떻게 하는가?

실무에서는 **Ground Truth가 확보된 후에** 재학습합니다.

| 도메인 | Ground Truth | 확보 시점 |
|--------|-------------|----------|
| **대출 심사** | 실제 상환/연체 결과 | 대출 실행 후 3~6개월 |
| **광고 CTR** | 실제 클릭 여부 | 광고 노출 직후 (수초~수분) |
| **이탈 예측** | 실제 이탈 여부 | 1~3개월 후 |
| **의료 진단** | 전문의 판독 결과 | 수일~수주 |

대출의 경우를 예로 들면:

```
1월: 모델이 "승인" 예측 -> 실제로 대출 실행
     |
     v
4월: 3개월 후, 이 고객이 실제로 잘 갚고 있는지 확인 가능
     -> 이 '실제 상환 결과'가 Ground Truth
     -> 이 데이터로 재학습해야 진짜 의미 있는 모델이 됨
```

### 실무 재학습 데이터 흐름

```
  예측 로그 (입력 피처만)          실제 결과 (Ground Truth)
  +-------------------+          +-------------------+
  | 나이, 연소득, ...  |          | 상환 완료 / 연체   |
  | (예측 시점에 기록)  |   JOIN   | (3~6개월 후 확보)  |
  +-------------------+  ------>  +-------------------+
                                          |
                                          v
                                  재학습용 데이터 완성
                                  (피처 + 정답 라벨)
```

### 정리

| | 이 실습 | 실무 |
|---|--------|------|
| **재학습 라벨** | 기존 모델의 예측값 | 실제 결과 (Ground Truth) |
| **라벨 확보 시점** | 즉시 | 도메인에 따라 수일~수개월 |
| **학습 목적** | 배포 파이프라인 실습 | 실제 모델 성능 개선 |

> **이 실습에서 배우는 것**: 재학습된 모델을 S3에 올리고 ECS에 반영하는 **운영 자동화 흐름**
>
> **실무에서 추가로 필요한 것**: Ground Truth 수집 파이프라인 (대출이면 상환 결과 DB 연동)

---

## Step 2: 데이터 로드

원본 학습 데이터와 운영 중 쌓인 예측 로그를 합쳐서 재학습합니다.
예측 로그에는 드리프트된 최신 데이터가 포함되어 있어, 이를 반영한 모델을 만들 수 있습니다.

In [ ]:
# 여기에 코드를 작성하세요


---

## Step 3: 기존 모델 vs 새 모델 학습 & 비교

기존 학습 데이터로 만든 모델과, 예측 로그를 포함한 새 데이터로 만든 모델의 AUC를 비교합니다.

In [ ]:
# 여기에 코드를 작성하세요


---

## Step 4: 새 모델 로컬 저장

성능이 확인되면 모델 파일을 로컬에 저장합니다.

In [ ]:
# 여기에 코드를 작성하세요


---

## Step 5: S3 버킷 생성

In [ ]:
# 여기에 코드를 작성하세요


---

## Step 6: 모델을 S3에 업로드

In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# 여기에 코드를 작성하세요


---

## Step 7: 앱 코드 수정 — S3에서 모델 로드

### model.py 수정 (load 메서드)

기존: 로컬 파일에서 로드
```python
self.pipeline = joblib.load('models/loan_pipeline.pkl')
```

변경: S3에서 다운로드 후 로드
```python
import boto3

def load(self, model_dir='models'):
    s3_bucket = os.environ.get('MODEL_S3_BUCKET')
    s3_prefix = os.environ.get('MODEL_S3_PREFIX', 'loan-api')

    if s3_bucket:
        # S3에서 모델 다운로드
        s3 = boto3.client('s3')
        os.makedirs(model_dir, exist_ok=True)

        for filename in ['loan_pipeline.pkl', 'label_encoders.pkl', 'feature_names.pkl']:
            s3_key = f'{s3_prefix}/{filename}'
            local_path = os.path.join(model_dir, filename)
            s3.download_file(s3_bucket, s3_key, local_path)
            logger.info(f'S3에서 모델 다운로드: s3://{s3_bucket}/{s3_key}')

    # 기존 로컬 로드 로직 (동일)
    self.pipeline = joblib.load(os.path.join(model_dir, 'loan_pipeline.pkl'))
    self.label_encoders = joblib.load(os.path.join(model_dir, 'label_encoders.pkl'))
    self.feature_names = joblib.load(os.path.join(model_dir, 'feature_names.pkl'))
```

### 핵심 포인트

- `MODEL_S3_BUCKET` 환경변수가 설정되면 S3에서 다운로드
- 환경변수가 없으면 기존처럼 로컬 파일에서 로드 (개발 환경 호환)
- 서버 시작 시(Lifespan) 한 번만 다운로드하므로 요청마다 S3를 호출하지 않음

---

## Step 8: Task Definition에 환경변수 추가

ECS Task Definition의 `containerDefinitions`에 S3 버킷 정보를 추가합니다.

```json
"environment": [
  {
    "name": "MODEL_S3_BUCKET",
    "value": "mlops-lab-models-123456789012"
  },
  {
    "name": "MODEL_S3_PREFIX",
    "value": "loan-api"
  }
]
```

### ECS Task Role에 S3 접근 권한 추가

ecsTaskRole (Task가 실행 중에 사용하는 역할)에 S3 읽기 권한이 필요합니다.

```bash
# ecsTaskRole에 S3 읽기 정책 연결
aws iam attach-role-policy \
  --role-name ecsTaskRole \
  --policy-arn arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess
```

**주의**: `ecsTaskExecutionRole`이 아니라 `ecsTaskRole`입니다.
- `ecsTaskExecutionRole`: ECS가 이미지 pull + 로그 전송에 사용
- `ecsTaskRole`: 컨테이너 내부 코드가 AWS 서비스에 접근할 때 사용

---

## Step 9: ECS 서비스 재시작 (모델 갱신)

S3에 새 모델을 업로드한 후, ECS 서비스를 재시작하면
새 컨테이너가 시작하면서 S3에서 최신 모델을 다운로드합니다.

```
S3에 새 pkl 업로드 → ECS force-new-deployment → 새 컨테이너 시작
                                                  → Lifespan에서 S3 다운로드
                                                  → 새 모델로 예측 서비스 시작
```

In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# 여기에 코드를 작성하세요


---

## Step 10: 배포 확인 — API 테스트

In [ ]:
# 여기에 코드를 작성하세요


---

## 운영 자동화 전체 흐름 정리

### 모델 업데이트 시나리오

```
1. [자동] PSI 모니터링 → 드리프트 감지 → Slack 알림
2. [사람] 알림 확인 → 재학습 필요 여부 판단
3. [사람] 이 노트북에서 재학습 → 새 모델 생성 → 성능 비교
4. [사람] S3에 업로드 (셀 실행)
5. [자동] Lambda가 S3 이벤트를 감지 → ECS force-new-deployment
6. [자동] 새 컨테이너가 S3에서 모델 다운로드 → 배포 완료
```

### 코드 변경 vs 모델 변경

| 변경 내용 | 필요한 작업 | 소요 시간 |
|-----------|-----------|----------|
| **코드 변경** | docker build → ECR push → ECS 배포 | 5~10분 |
| **모델만 변경** | S3 업로드 → ECS 재시작 | 1~3분 |

모델과 코드를 분리하면 **모델 업데이트가 훨씬 빠르고 간단**합니다.

### 자동화 수준별 비교

| 수준 | 방식 | 이 과정 |
|------|------|--------|
| Level 0 | 수동 (ssh → 파일 교체) | |
| Level 1 | S3 업로드 + ECS 재시작 (CLI) | **이번 실습** |
| Level 2 | S3 이벤트 → Lambda → ECS 재시작 (자동) | 실무 권장 |
| Level 3 | Airflow/Step Functions 파이프라인 | 대규모 MLOps |

---

## Step 11: 진짜 자동화 — S3 업로드만 하면 ECS 자동 재시작

현재까지는 S3 업로드 후 `update-service`를 **수동으로** 실행해야 합니다.
진짜 자동화는 **S3에 모델이 올라오면 Lambda가 자동으로 ECS를 재시작**하는 것입니다.

```
사람: S3에 pkl 업로드  →  [자동] S3 이벤트 트리거
                          →  [자동] Lambda 함수 실행
                          →  [자동] ECS force-new-deployment
                          →  [자동] 새 컨테이너가 S3에서 모델 다운로드
                          →  [자동] 배포 완료
```

### Lambda 함수 코드

```python
import boto3

def lambda_handler(event, context):
    ecs = boto3.client('ecs')

    ecs.update_service(
        cluster='mlops-cluster',
        service='loan-api-service',
        forceNewDeployment=True,
    )

    bucket = event['Records'][0]['s3']['bucket']['name']
    key = event['Records'][0]['s3']['object']['key']
    print(f'모델 업데이트 감지: s3://{bucket}/{key}')
    print(f'ECS 서비스 재시작 요청 완료')

    return {'statusCode': 200, 'body': 'ECS redeployment triggered'}
```

### 설정 순서 (AWS 콘솔)

1. **Lambda 함수 생성**: 위 코드를 Lambda에 등록 (Python 3.10, 이름: `model-update-trigger`)
2. **Lambda 권한**: `AmazonECS_FullAccess` 정책을 Lambda 역할에 연결
3. **S3 이벤트 트리거**: S3 버킷 → Properties → Event notifications → Lambda 연결
   - Event type: `PUT` (파일 업로드)
   - Prefix: `loan-api/` (이 폴더에 업로드될 때만)
   - Suffix: `.pkl` (pkl 파일만)
   - Destination: Lambda 함수 선택

### 설정 후 흐름

```bash
# 사람이 할 일은 이것 하나뿐:
aws s3 cp models/loan_pipeline.pkl s3://mlops-lab-models-{ACCOUNT_ID}/loan-api/

# 나머지는 전부 자동:
# → S3 이벤트 발생 → Lambda 실행 → ECS 재시작 → 새 모델 반영
```

In [ ]:
# 여기에 코드를 작성하세요


---

## 빠른 참조: 모델 업데이트 CLI 명령어

```bash
# 1. 재학습 후 모델 파일을 S3에 업로드
aws s3 cp models/loan_pipeline.pkl s3://mlops-lab-models-{ACCOUNT_ID}/loan-api/
aws s3 cp models/label_encoders.pkl s3://mlops-lab-models-{ACCOUNT_ID}/loan-api/
aws s3 cp models/feature_names.pkl s3://mlops-lab-models-{ACCOUNT_ID}/loan-api/

# 2. ECS 서비스 재시작 (새 컨테이너가 S3에서 최신 모델 다운로드)
aws ecs update-service \
  --cluster mlops-cluster \
  --service loan-api-service \
  --force-new-deployment

# 3. 배포 완료 대기
aws ecs wait services-stable \
  --cluster mlops-cluster \
  --services loan-api-service

# 4. 테스트
curl http://{퍼블릭IP}:8000/health
```

**이 4줄이면 모델 업데이트 완료입니다.**

---

## Lambda 트리거 설정 — 단계별 AWS 콘솔 가이드

S3에 모델이 업로드되면 Lambda가 자동으로 ECS를 재시작하도록 설정합니다.

### Step 1: Lambda 함수 생성

1. AWS 콘솔 → **Lambda** → **함수 생성**
2. 설정:
   - 함수 이름: `model-update-trigger`
   - 런타임: Python 3.10
   - 아키텍처: x86_64
   - 실행 역할: **새 역할 생성** (기본 Lambda 권한)
3. **함수 생성** 클릭
4. 코드 탭에서 아래 코드 붙여넣기:

```python
import boto3

def lambda_handler(event, context):
    ecs = boto3.client('ecs')
    ecs.update_service(
        cluster='mlops-cluster',
        service='loan-api-service',
        forceNewDeployment=True,
    )
    bucket = event['Records'][0]['s3']['bucket']['name']
    key = event['Records'][0]['s3']['object']['key']
    print(f'Model update detected: s3://{bucket}/{key}')
    print(f'ECS redeployment triggered')
    return {'statusCode': 200, 'body': 'ECS redeployment triggered'}
```

5. **Deploy** 클릭

### Step 2: Lambda 실행 역할에 ECS 권한 추가

1. Lambda 함수 → **구성** 탭 → **권한** → 역할 이름 클릭
2. IAM 콘솔이 열림 → **권한 추가** → **정책 연결**
3. `AmazonECS_FullAccess` 검색 → 체크 → **정책 연결**

### Step 3: S3 이벤트 트리거 설정

1. S3 콘솔 → 버킷 선택 (`mlops-lab-models-{ACCOUNT_ID}`)
2. **속성** 탭 → **이벤트 알림** → **이벤트 알림 생성**
3. 설정:
   - 이벤트 이름: `model-upload-trigger`
   - 접두사(Prefix): `loan-api/`
   - 접미사(Suffix): `.pkl`
   - 이벤트 유형: `s3:ObjectCreated:Put` 체크
   - 대상: **Lambda 함수** → `model-update-trigger` 선택
4. **변경 사항 저장**

### Step 4: 테스트

```bash
# S3에 모델 업로드 (이것만 하면 자동으로 ECS 재시작!)
aws s3 cp models/loan_pipeline.pkl s3://mlops-lab-models-{ACCOUNT_ID}/loan-api/

# Lambda 실행 확인
# CloudWatch → 로그 그룹 → /aws/lambda/model-update-trigger

# ECS 재시작 확인
aws ecs describe-services --cluster mlops-cluster --services loan-api-service \
  --query 'services[0].deployments'
```

### 주의사항

- Lambda 타임아웃을 **30초 이상**으로 설정하세요 (기본 3초는 너무 짧음)
- Lambda와 S3 버킷이 **같은 리전**에 있어야 합니다
- 이벤트 알림에서 Suffix를 `.pkl`로 설정하면 **pkl 파일만** 트리거됩니다

---

## 배포 검증 방법 — 새 모델이 제대로 반영되었는지 확인

### 1. ECS 배포 이벤트 확인

```bash
aws ecs describe-services \
  --cluster mlops-cluster \
  --services loan-api-service \
  --query 'services[0].{
    Deployments: deployments[*].{
      Status: status,
      DesiredCount: desiredCount,
      RunningCount: runningCount,
      CreatedAt: createdAt
    },
    Events: events[:3].message
  }'

# 정상 배포 완료 시:
# - PRIMARY 배포의 RunningCount == DesiredCount
# - Events에 "has reached a steady state" 메시지
```

### 2. CloudWatch 로그 확인

```bash
# 최근 로그 스트림 확인
aws logs describe-log-streams \
  --log-group-name /ecs/loan-api \
  --order-by LastEventTime \
  --descending \
  --limit 3

# 정상이면 이런 로그가 보임:
# "S3에서 모델 다운로드: s3://mlops-lab-models-.../loan-api/loan_pipeline.pkl"
# "모델 로드 완료"
```

### 3. 검증 체크리스트

- [ ] ECS 서비스가 steady state에 도달했는가?
- [ ] CloudWatch 로그에 S3 다운로드 성공 메시지가 있는가?
- [ ] /health 응답에서 model_loaded가 true인가?
- [ ] /predict 응답이 정상적으로 오는가?

---

## Troubleshooting — 자주 발생하는 에러와 해결법

### 1. S3 Permission Denied (AccessDenied)

```
botocore.exceptions.ClientError: An error occurred (AccessDenied)
when calling the PutObject operation: Access Denied
```

**원인**: IAM 권한 부족
**해결**:
```bash
# 노트북 실행 환경의 IAM 역할에 S3 쓰기 권한 추가
aws iam attach-role-policy \
  --role-name {역할이름} \
  --policy-arn arn:aws:iam::aws:policy/AmazonS3FullAccess

# ECS 컨테이너가 S3에서 읽지 못하는 경우
# → ecsTaskRole (ecsTaskExecutionRole 아님!)에 S3ReadOnly 추가
aws iam attach-role-policy \
  --role-name ecsTaskRole \
  --policy-arn arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess
```

---

### 2. 모델 버전 불일치 (Model Version Mismatch)

```
ValueError: feature_names mismatch / sklearn version mismatch
```

**원인**: 학습 환경과 서빙 환경의 패키지 버전이 다름
**해결**:
- `requirements.txt`에 패키지 버전을 **정확히 고정**
  ```
  scikit-learn==1.4.0
  xgboost==2.0.3
  joblib==1.3.2
  ```
- 학습 환경과 Docker 이미지가 **같은 requirements.txt** 사용

---

### 3. ECS OOM (Out of Memory) — 새 모델이 너무 클 때

```
Container killed due to memory limit exceeded
Essential container in task exited
```

**원인**: 새 모델이 기존보다 크거나, 메모리 사용량이 증가
**해결**:
```bash
# Task Definition의 메모리 제한 확인/증가
aws ecs describe-task-definition \
  --task-definition loan-api \
  --query 'taskDefinition.containerDefinitions[0].{memory: memory, memoryReservation: memoryReservation}'

# 해결: Task Definition에서 memory를 512 → 1024로 증가
```

---

### 4. Lambda가 트리거되지 않음

**확인 순서**:
1. S3 이벤트 알림이 정상 설정되었는가? (Prefix: `loan-api/`, Suffix: `.pkl`)
2. Lambda 함수가 존재하는가?
3. Lambda 역할에 ECS 권한이 있는가?
4. CloudWatch에서 Lambda 로그 확인

```bash
# Lambda 로그 확인
aws logs tail /aws/lambda/model-update-trigger --since 1h

# S3 이벤트 알림 확인
aws s3api get-bucket-notification-configuration \
  --bucket mlops-lab-models-{ACCOUNT_ID}
```

---

### 5. ECS 재시작 후 컨테이너가 계속 실패

```
Task stopped: Essential container in task exited
```

**확인 순서**:
1. CloudWatch 로그에서 에러 메시지 확인
2. S3에 모델 파일이 정상 업로드되었는가?
3. Task Role에 S3 읽기 권한이 있는가?
4. 모델 파일이 손상되지 않았는가? (로컬에서 `joblib.load`로 테스트)